# Automated variable selection using Factor Analysis

Inspired by https://doi.org/10.1080/10095020.2019.1621549


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from factor_analyzer import FactorAnalyzer
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.decomposition import PCA

## Preprocess the data
Remove unnecessary column, devide the data by total population and standardize it.

In [ ]:
data_relative = gpd.read_parquet(
    "/data/uscuni-restricted/geodem/05_data/_data_all.parquet"
)

In [ ]:
data_relative = data_relative.drop(
    columns=["Obyvatelstvo celkem", "Hospodařící domácnosti v bytech celkem"]
)

In [ ]:
var = data_relative.columns[13:]

In [ ]:
X_processed = data_relative[var]
feature_names = list(X_processed.columns)

In [ ]:
X_processed

# Factor analysis


In [ ]:
#  Determine the number of factors using Kaiser's Criterion
fa_initial = FactorAnalyzer()
fa_initial.fit(X_processed)
eigenvalues, _ = fa_initial.get_eigenvalues()
N = np.sum(eigenvalues >= 1)

print(f"Using Kaiser's Criterion: Selected N = {N} factors.")

In [ ]:
# Plot factors and eigenvalues
plt.figure(figsize=(20, 10))
plt.plot(range(1, len(eigenvalues) + 1), eigenvalues, marker="o")

plt.axhline(1, color="r", linestyle="--")

plt.xlabel("Factors")
plt.ylabel("Eigenvalue")
plt.grid()
plt.show()

### Communality Check

In [ ]:
# Initialize and fit Factor Analysis
fa_n = FactorAnalyzer(n_factors=N)
fa_n.fit(X_processed)
# Get the proportion of variance explained by factors
C1 = fa_n.get_communalities()
C2 = np.mean(C1)
# Keep only variables whose communality is at least the mean
communality_mask = C1 >= C2
vars_after_communality = list(X_processed.columns[communality_mask])
print(f"\nKept {len(vars_after_communality)} variables after communality check.")

### Minimum Spanning Tree Correlation Check

In [ ]:
# Select subset of processed variables
corr_matrix = X_processed[vars_after_communality].corr()

# Define correlation threshold
correlation_threshold = 0.75

# Build a distance-like graph: distance = 1 - |correlation|
graph = 1 - np.abs(corr_matrix)

# Keep only edges above the threshold
graph[graph > (1 - correlation_threshold)] = 0

# Convert graph to a sparse matrix and compute MST
mst = minimum_spanning_tree(csr_matrix(graph)).toarray()

# Extract pairs of variables (edges)
edges = np.argwhere(mst > 0)

# Initialize a set to store variables to remove
vars_to_remove = set()

In [ ]:
if edges.size > 0:
    # Compute the "degree" for each variable in MST
    degrees = np.sum(mst > 0, axis=0) + np.sum(mst > 0, axis=1)

    for u, v in edges:
        var1 = vars_after_communality[u]
        var2 = vars_after_communality[v]

        # Remove the variable with fewer connections
        if degrees[u] < degrees[v]:
            vars_to_remove.add(var1)
        else:
            vars_to_remove.add(var2)

    # Keep only variables not flagged for removal
    vars_keep = [v for v in vars_after_communality if v not in vars_to_remove]
    print(f"Removed {len(vars_to_remove)} variables via MST: {list(vars_to_remove)}")
else:
    print("No highly correlated pairs found.")
    vars_keep = vars_after_communality

In [ ]:
fa_final_variables = vars_keep
print(
    f"\nFinal Set of {len(fa_final_variables)} Variables based on {N} components and correlation filtering:"
)

fa_final_variables

In [ ]:
# Determine the number of components using Kaiser's Criterion (Harsh Threshold)
pca_initial = PCA().fit(X_processed)
eigenvalues = pca_initial.explained_variance_
N = np.sum(eigenvalues >= 1)

print(f"Using Kaiser's Criterion: Selected N = {N} components.")

In [ ]:
from sklearn.decomposition import PCA

# --- STAGE 3: PCA Variable Contribution ---
# Initialize and fit PCA with the N components identified in Stage 2
pca_n = PCA(n_components=N)
pca_n.fit(X_processed)

# Get the eigenvalues (explained variance) for the N components
eigenvalues = pca_n.explained_variance_

# pca_n.components_ contains the eigenvectors (loadings)
# Shape is (n_components, n_features), so we transpose to (n_features, n_components)
loadings = pca_n.components_.T

# According to the flowchart:
# Variable Contribution = Sum of (squared loading * eigenvalue) across N components
# This represents the total variance of the variable explained by the retained PCs
C1 = np.sum((loadings**2) * eigenvalues, axis=1)

# C2 = Mean contribution
C2 = np.mean(C1)

# Keep only variables whose contribution is at least the average
communality_mask = C1 >= C2
vars_after_communality = list(X_processed.columns[communality_mask])

print(f"\nKept {len(vars_after_communality)} variables after contribution check.")
print(f"Mean contribution threshold (C2): {C2:.4f}")

In [ ]:
# Select subset of processed variables
corr_matrix = X_processed[vars_after_communality].corr()

# Define correlation threshold
correlation_threshold = 0.75

# Build a distance-like graph: distance = 1 - |correlation|
graph = 1 - np.abs(corr_matrix)

# Keep only edges above the threshold
graph[graph > (1 - correlation_threshold)] = 0

# Convert graph to a sparse matrix and compute MST
mst = minimum_spanning_tree(csr_matrix(graph)).toarray()

# Extract pairs of variables (edges)
edges = np.argwhere(mst > 0)

# Initialize a set to store variables to remove
vars_to_remove = set()

In [ ]:
if edges.size > 0:
    # Compute the "degree" for each variable in MST
    degrees = np.sum(mst > 0, axis=0) + np.sum(mst > 0, axis=1)

    for u, v in edges:
        var1 = vars_after_communality[u]
        var2 = vars_after_communality[v]

        # Remove the variable with fewer connections
        if degrees[u] < degrees[v]:
            vars_to_remove.add(var1)
        else:
            vars_to_remove.add(var2)

    # Keep only variables not flagged for removal
    vars_keep = [v for v in vars_after_communality if v not in vars_to_remove]
    print(f"Removed {len(vars_to_remove)} variables via MST: {list(vars_to_remove)}")
else:
    print("No highly correlated pairs found.")
    vars_keep = vars_after_communality

In [ ]:
pca_final_variables = vars_keep
print(
    f"\nFinal Set of {len(pca_final_variables)} Variables based on {N} components and correlation filtering:"
)

pca_final_variables

In [ ]:
set_a = set(fa_final_variables)
set_b = set(pca_final_variables)

# Variables in A but NOT in B
only_in_a = set_a - set_b

# Variables in B but NOT in A
only_in_b = set_b - set_a

print(f"Variables only in List A ({len(only_in_a)}):")
for var in only_in_a:
    print(var)
print(f"\nVariables only in List B ({len(only_in_b)}):")
for var in only_in_b:
    print(var)